In [1]:
import os
import shutil
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import time
import openpyxl
from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo
import pandas as pd
import xlwings as xw
import numpy as np

In [2]:
date_str = datetime.now().strftime('%Y%m%d')
#C:\Users\stephalin\Downloads
download_path = input("Enter path to folder where all raw sources files are downloaded: ")
raw_Cloud = os.path.join(download_path, input("Enter ESC Cloud source file name: ") + "." + input("xlsx or csv?"))

Enter path to folder where all raw sources files are downloaded:  C:\Users\stephalin\Downloads
Enter ESC Cloud source file name:  ESCIDExport 11-25
xlsx or csv? xlsx


In [4]:
raw_CostPool = os.path.join(download_path,"TBM ESC Cost Pool.xlsx")
raw_T = os.path.join(download_path,"TBM ESC Tower.xlsx")

In [8]:
df = pd.read_excel(raw_T, sheet_name="TBM ESC Tower", header=1)
#OCIO_Admin = pd.read_excel(raw_T, sheet_name="OCIO ADMIN CODES", header=0)

#df = df.iloc[1:].reset_index(drop=True)
TowerRaw_len = len(df)

# Find the index after "Total Amount"
total_amt_idx = df.columns.get_loc("Amount") + 0

# Insert in reverse order
df.insert(total_amt_idx, "Service Line", "")
df.insert(total_amt_idx, "Office", "")
df.insert(total_amt_idx, "In OCIO?", "")
df['In OCIO?'] = df['Admin Code'].str.contains('HCAJR', na=False)


In [9]:
start4 = time.time()
#build TBM Tower ratios (pivot, filter for FY25)
tower_FY = df[(df['Fiscal Year'] == 2026) & (df['Amount'] != 0)]
tower_pivot = tower_FY.pivot_table(
    index=["ESC ID", "Towers"],
    values=["Amount"],
    aggfunc="sum"
).reset_index()

tower_pivot['Ratio'] = (
    tower_pivot.groupby('ESC ID')['Amount'].transform(lambda x: x / x.sum())
)

#tower_pivot

In [10]:
cp_df = pd.read_excel(raw_CostPool, sheet_name="TBM ESC Cost Pool", header=1)
#df = df.iloc[1:].reset_index(drop=True)
CostPoolRaw_len = len(cp_df)

# Find the index after "Total Amount"
total_amt_idx = cp_df.columns.get_loc("CP Amount") + 1

# Insert in reverse order
cp_df.insert(total_amt_idx, "Service Line", "")
cp_df.insert(total_amt_idx, "Office", "")
cp_df.insert(total_amt_idx, "In OCIO?", "")

In [11]:
#build TBM CP ratios (pivot, filter for FY25)
cp_FY = cp_df[(cp_df['Fiscal Year'] == 2026) & (cp_df['CP Amount'] != 0)]

cp_pivot = cp_FY.pivot_table(
    index=["ESC ID", "Cost Pools"],
    values=["CP Amount"],
    aggfunc="sum"
).reset_index()

cp_pivot['Ratio'] = (
    cp_pivot.groupby('ESC ID')['CP Amount']
    .transform(lambda x: x / x.sum())
)

#cp_pivot

In [12]:
#pivot export on cloud data (actuals)
Cloud_df = pd.read_excel(raw_Cloud, sheet_name="Export", header=0)
Cloud_df = Cloud_df[Cloud_df['ESCID'].notnull()]
Cloud_len = len(Cloud_df)

In [13]:
#user to insert actual column name for billing month
Actuals = Cloud_df.pivot_table(
    index=["ESCID", "BILLING MONTH"],
    values=["COST"],
    aggfunc="sum"
).reset_index()

#Actuals

In [14]:
#create tower breakdown (join actuals and ratios)
tower_merge = Actuals.merge(tower_pivot, left_on='ESCID', right_on='ESC ID', how='left')
#ESCID = 0 + N/A mappings
tower_merge['Towers'] = tower_merge.apply(
    lambda x: 'No ESC ID Tagged' if x['ESCID'] == 0 
    else ('No CPIC Tower Projection' if pd.isna(x['Towers']) else x['Towers']), axis=1
)
#Ratio N/A mappings
tower_merge['Ratio'] = tower_merge['Ratio'].fillna(1)

tower_merge = tower_merge.drop(columns=['ESC ID','Amount'])
tower_merge['New Expenditures'] = tower_merge['COST'] * tower_merge['Ratio']
#tower_merge[tower_merge.ESCID == 3476]
#tower_merge


In [15]:
#create CP breakdown (join actuals and ratios)
cp_merge = Actuals.merge(cp_pivot, left_on='ESCID', right_on='ESC ID', how='left')
#ESCID = 0 + N/A mappings
cp_merge['Cost Pools'] = cp_merge.apply(
    lambda x: 'No ESC ID Tagged' if x['ESCID'] == 0 
    else ('No CPIC Cost Pool Projection' if pd.isna(x['Cost Pools']) else x['Cost Pools']), axis=1
)
#Ratio N/A mappings
cp_merge['Ratio'] = cp_merge['Ratio'].fillna(1)

cp_merge = cp_merge.drop(columns=['ESC ID','CP Amount'])
cp_merge['New Expenditures'] = cp_merge['COST'] * cp_merge['Ratio']
#cp_merge[cp_merge.ESCID == 2890]

In [16]:
"""for cloud, tabs Tower Breakdowns + CP Breakdown should have col B as BILLING MONTH"""
cp_merge.head()

,ESCID,BILLING MONTH,COST,Cost Pools,Ratio,New Expenditures
0,0,2025-10-01,288096.54,No ESC ID Tagged,1.000000,288096.540000
1,0,2025-11-01,280454.03,No ESC ID Tagged,1.000000,280454.030000
2,50,2025-10-01,2919.26,External Labor,0.382353,1116.187647
3,50,2025-10-01,2919.26,Internal Labor,0.073529,214.651471
4,50,2025-10-01,2919.26,Other,0.029412,85.860588


In [17]:
dst_Cloud = download_path+"\ESCIDdata export_"+date_str+".xlsx"
print("Copying contents of "+raw_Cloud+" into "+dst_Cloud)
with pd.ExcelWriter(dst_Cloud, engine='openpyxl', mode='w') as writer:
    Cloud_df.to_excel(writer, sheet_name='Export', index=False)
    Actuals.to_excel(writer, sheet_name='Actuals', index=False)
    cp_pivot.to_excel(writer, sheet_name='CP Ratios', index=False)
    tower_pivot.to_excel(writer, sheet_name='Tower Ratios', index=False)
    tower_merge.to_excel(writer, sheet_name='Tower Breakdowns', index=False)
    cp_merge.to_excel(writer, sheet_name='CP Breakdown', index=False)

Copying contents of C:\Users\stephalin\Downloads\ESCIDExport 11-25.xlsx into C:\Users\stephalin\Downloads\ESCIDdata export_20251209.xlsx


In [18]:
end4 = time.time()
print(f"Elapsed time: {(end4 - start4)} seconds for Cloud data")
print(f"{Cloud_len} rows for Cloud Data")

Elapsed time: 34.09725332260132 seconds for Cloud data
46096 rows for Cloud Data
